<a href="https://colab.research.google.com/github/OjasTamhankar/AML-Lab/blob/main/exp10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [2]:
# Keep only top 10,000 frequent words
vocab_size = 10000

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 25000
Test samples: 25000


In [3]:
max_length = 200

X_train = pad_sequences(X_train, maxlen=max_length)
X_test = pad_sequences(X_test, maxlen=max_length)

In [4]:
model = Sequential()

model.add(Embedding(vocab_size, 128, input_length=max_length))
model.add(LSTM(128))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 184s 576ms/step - accuracy: 0.7870 - loss: 0.4413 - val_accuracy: 0.8250 - val_loss: 0.3946
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 164s 525ms/step - accuracy: 0.8951 - loss: 0.2661 - val_accuracy: 0.8674 - val_loss: 0.3288
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 165s 527ms/step - accuracy: 0.9275 - loss: 0.1962 - val_accuracy: 0.8512 - val_loss: 0.3548
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 167s 535ms/step - accuracy: 0.9457 - loss: 0.1497 - val_accuracy: 0.8518 - val_loss: 0.4477
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 200s 530ms/step - accuracy: 0.9618 - loss: 0.1104 - val_accuracy: 0.8582 - val_loss: 0.4684


In [6]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 82s 104ms/step - accuracy: 0.8505 - loss: 0.4855
Test Accuracy: 0.8505200147628784


In [7]:
word_index = imdb.get_word_index()

def encode_review(text):
    words = text.lower().split()
    encoded = []

    for word in words:
        if word in word_index and word_index[word] < vocab_size:
            encoded.append(word_index[word] + 3)
        else:
            encoded.append(2)  # unknown word

    return pad_sequences([encoded], maxlen=max_length)

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [12]:
def predict_sentiment(text):
    encoded = encode_review(text)
    prediction = model.predict(encoded)[0][0]

    if prediction > 0.3:
        return "Positive: +"
    else:
        return "Negative: -"

In [13]:
print(predict_sentiment("This movie was amazing and very interesting"))
print(predict_sentiment("The movie was boring and too long"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
Positive: +
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
Negative: -
